# 위험중립 GBM–Monte Carlo 구현 설명

이 문서는 유럽형 옵션을 가격화하기 위해 구현한 **정확한 위험중립 GBM 만기 표본추출**과 **Monte Carlo 가격 추정** 코드의 전체 구조를 설명한다. 목표는 각 파일의 책임, 데이터가 이동하는 순서, 수식이 코드에 대응되는 위치, 입력 검증과 테스트 의도를 한눈에 연결하는 것이다.

관련 파일은 다음과 같다.

- GBM 만기주가 표본추출: [gbm.py](../src/option_pricing_volatility/processes/gbm.py)
- Monte Carlo 가격 추정: [monte_carlo.py](../src/option_pricing_volatility/simulation/monte_carlo.py)
- 단위 테스트: [test_monte_carlo.py](../tests/test_monte_carlo.py)
- 연구 노트북: [03_01_GBM_MC_model.ipynb](../notebooks/03_gbm_mc/03_01_GBM_MC_model.ipynb)
- 금융·수치 계약: [model_contracts.md](model_contracts.md)

향후 연구 질문은 동일한 $S,K,T,r,q,\sigma$ 가정에서 CRR 가격과 GBM–Monte Carlo 가격이 BSM 가격으로 수렴하는지 확인하는 것이다. 현재 구현은 그중 terminal GBM sampling과 Monte Carlo 추정량까지만 담당하며, BSM 함수와 최종 비교 그래프는 포함하지 않는다.

## 1. 전체 구조와 호출 흐름

<pre>
연구 노트북 / 사용자
        |
        |  spot, strike, T, r, sigma, q, n_paths, option_type, seed
        v
simulation/monte_carlo.py : mc_price()
        |
        |-- 입력 전체 검증
        |-- T=0 또는 sigma=0 결정론적 경계 처리
        |-- seed로 numpy.random.Generator 생성
        |
        v
processes/gbm.py : sample_gbm_terminal()
        |
        |-- 정확한 위험중립 GBM 만기분포에서 S_T 벡터 생성
        v
shape (n_paths,), dtype float64 배열
        |
        v
call/put payoff -> 현재가치 할인 -> 평균 / 표준오차 / 신뢰구간
        |
        v
MonteCarloResult
</pre>

역할을 두 모듈로 나눈 이유는 **확률과정 표본추출**과 **금융상품 payoff 및 통계 추정**을 분리하기 위해서다.

| 모듈 | 책임 | 알지 못하는 것 |
|---|---|---|
| <code>processes/gbm.py</code> | 위험중립 GBM의 $S_T$ 표본 벡터 생성 | strike, call/put payoff, 할인 통계 |
| <code>simulation/monte_carlo.py</code> | 옵션 payoff, 할인, 가격·표준오차·신뢰구간 계산 | GBM 난수 변환의 내부 구현 |
| 연구 노트북 | 파라미터 설정, 반복 실험, tidy DataFrame 구성 | 가격 수식의 재구현 |

따라서 GBM 표본추출 로직은 다른 terminal-payoff 연구에서도 재사용할 수 있고, 연구 노트북은 패키지 함수를 호출하는 orchestration 역할만 유지한다.

## 2. 두 공개 함수의 인터페이스

### 2.1 terminal GBM sampler

<pre><code>sample_gbm_terminal(
    spot,
    maturity,
    rate,
    volatility,
    n_paths,
    dividend_yield=0.0,
    *,
    rng,
) -> NDArray[np.float64]</code></pre>

<code>rng</code>는 keyword-only이며 반드시 <code>numpy.random.Generator</code>여야 한다. 반환값은 길이가 <code>n_paths</code>인 1차원 <code>float64</code> 배열이다.

### 2.2 Monte Carlo pricer

<pre><code>mc_price(
    spot,
    strike,
    maturity,
    rate,
    volatility,
    n_paths,
    option_type,
    dividend_yield=0.0,
    *,
    seed,
    confidence_level=0.95,
) -> MonteCarloResult</code></pre>

<code>seed</code>도 keyword-only이고 기본값이 없다. 즉, 호출자가 매번 난수 seed를 명시해야 한다.

| 입력 | 금융·수치 의미 |
|---|---|
| <code>spot</code> | 현재 기초자산 가격 $S>0$ |
| <code>strike</code> | 행사가격 $K>0$; pricer에만 필요 |
| <code>maturity</code> | 잔존만기 $T\ge 0$, 연 단위 |
| <code>rate</code> | 연속복리 무위험이자율 $r$ |
| <code>volatility</code> | 연율 변동성 $\sigma\ge 0$ |
| <code>dividend_yield</code> | 연속복리 배당수익률 $q$ |
| <code>n_paths</code> | 표본 수 $M\ge 2$ |
| <code>option_type</code> | 정확히 <code>call</code> 또는 <code>put</code> |
| <code>seed</code> | 0 이상의 정수; 결과 객체에도 기록 |
| <code>confidence_level</code> | $0<c<1$; 기본값 0.95 |

## 3. <code>sample_gbm_terminal()</code>의 계산 구조

위험중립측도에서 연속배당수익률 $q$를 갖는 GBM은 다음과 같다.

$$
dS_t=(r-q)S_t\,dt+\sigma S_t\,dW_t
$$

이 확률미분방정식의 정확한 만기분포를 사용하면 각 표본은

$$
S_T^{(i)}
=S_0\exp\left[
\left(r-q-\frac12\sigma^2\right)T
+\sigma\sqrt{T}Z_i
\right],
\qquad Z_i\sim N(0,1)
$$

이다. 구현은 이 식을 NumPy 배열 연산으로 한 번에 평가한다.

계산 순서는 다음과 같다.

1. 모든 스칼라가 bool이 아닌 유한한 실수인지 확인한다.
2. $S>0$, $T\ge0$, $\sigma\ge0$, <code>n_paths >= 2</code>, 명시적 <code>Generator</code>를 확인한다.
3. $T=0$이면 모든 원소가 $S_0$인 배열을 반환한다.
4. drift $\left(r-q-\frac12\sigma^2\right)T$를 계산한다.
5. $\sigma=0$이면 모든 원소가 $S_0e^{(r-q)T}$인 결정론적 배열을 반환한다.
6. 일반적인 경우 <code>rng.standard_normal(n_paths)</code>로 $Z$ 벡터를 한 번 생성하고 terminal-price 벡터를 계산한다.

시간 전체 경로나 중간 시점 가격을 만들지 않으므로 시간복잡도와 추가 메모리는 모두 $O(M)$이다. Euler–Maruyama 근사를 사용하지 않기 때문에 time-step discretization error도 없다. 남는 오차는 옵션가격을 유한 표본평균으로 추정할 때 발생하는 Monte Carlo sampling error다.

## 4. <code>mc_price()</code>의 가격 계산 흐름

일반적인 $T>0$, $\sigma>0$ 호출은 다음 순서로 진행된다.

1. pricer가 자신의 모든 입력을 검증하고 Python/NumPy 정수형을 일관된 <code>float</code>와 <code>int</code>로 정규화한다.
2. <code>np.random.default_rng(seed)</code>로 이 호출 전용 Generator를 만든다.
3. Generator를 <code>sample_gbm_terminal()</code>에 전달해 $S_T$ 배열을 받는다.
4. <code>np.maximum</code>으로 모든 경로의 call 또는 put payoff를 벡터화해 계산한다.
5. payoff 배열 전체에 $e^{-rT}$를 곱해 현재가치 표본 $Y_i$를 만든다.
6. 평균, 표준오차, 정규근사 신뢰구간을 계산해 <code>MonteCarloResult</code>로 묶는다.

경로별 할인 payoff는 다음과 같다.

$$
Y_i^{\text{call}}=e^{-rT}\max(S_T^{(i)}-K,0)
$$

$$
Y_i^{\text{put}}=e^{-rT}\max(K-S_T^{(i)},0)
$$

경로 하나의 payoff $Y_i$와 옵션가격 추정량 $\hat V_M$은 서로 다른 값이다. 옵션가격은 할인 payoff들의 표본평균이다.

$$
\hat V_M=\frac1M\sum_{i=1}^{M}Y_i
$$

payoff 생성, 할인, 통계 계산은 모두 1차원 배열에서 이루어지며 전체 경로나 범용 simulation framework를 만들지 않는다.

## 5. 표준오차와 신뢰구간

할인 payoff의 표본표준편차는 Bessel 보정을 적용한 <code>ddof=1</code>을 사용한다.

$$
s_Y=\sqrt{\frac{1}{M-1}\sum_{i=1}^{M}(Y_i-\hat V_M)^2}
$$

표본평균의 표준오차는

$$
SE(\hat V_M)=\frac{s_Y}{\sqrt M}
$$

이다. confidence level을 $c$라고 하면 양쪽 꼬리확률과 임계값은

$$
\alpha_{\text{tail}}=\frac{1-c}{2},
\qquad
z_c=-\Phi^{-1}(\alpha_{\text{tail}})
$$

이고 신뢰구간은

$$
\left[
\hat V_M-z_cSE(\hat V_M),
\hat V_M+z_cSE(\hat V_M)
\right]
$$

이다. 기본 $c=0.95$에서는 $z_c\approx1.96$이다. 구현은 SciPy 없이 표준 라이브러리 <code>statistics.NormalDist</code>로 임계값을 계산한다. 하단 꼬리확률에서 임계값을 구하는 방식은 $c$가 1에 매우 가까울 때 상단 누적확률이 부동소수점 1.0으로 반올림되는 문제를 피한다.

가격이나 신뢰구간 경계를 no-arbitrage bound 또는 0으로 clipping하지 않는다. 따라서 sampling error가 큰 작은 표본에서는 call/put 신뢰구간의 하단이 음수가 될 수도 있으며, 이는 숨기지 않는 추정 불확실성이다.

## 6. 결과 객체와 결정론적 경계

<code>MonteCarloResult</code>는 frozen dataclass이며 다음 여섯 필드만 가진다.

| 필드 | 의미 |
|---|---|
| <code>price</code> | 할인 payoff의 표본평균 또는 결정론적 가격 |
| <code>standard_error</code> | 표본평균의 표준오차 |
| <code>ci_low</code> | 양측 신뢰구간 하단 |
| <code>ci_high</code> | 양측 신뢰구간 상단 |
| <code>n_paths</code> | 요청한 표본 수 |
| <code>seed</code> | 해당 추정에 사용한 seed |

frozen 객체이므로 계산 후 필드가 우연히 변경되지 않으며, 동일 seed 결과를 객체 전체 동등성으로 비교할 수 있다. <code>slots=True</code>는 이 작은 고정 구조에 불필요한 인스턴스 dictionary를 만들지 않는다.

### $T=0$

난수표본을 만들지 않고 즉시 내재가치를 반환한다.

$$
C=\max(S-K,0),
\qquad
P=\max(K-S,0)
$$

### $\sigma=0$, $T>0$

위험중립 주가가 결정론적이므로 할인된 payoff를 직접 반환한다.

$$
C=\max(Se^{-qT}-Ke^{-rT},0)
$$

$$
P=\max(Ke^{-rT}-Se^{-qT},0)
$$

두 경계에서 <code>standard_error=0</code>이고 <code>ci_low=price=ci_high</code>다. 내부 helper <code>_deterministic_result()</code>가 이 공통 결과 생성을 담당한다. 입력 검증은 경계 반환보다 먼저 수행하므로 $T=0$ 또는 $\sigma=0$이어도 잘못된 <code>n_paths</code>, seed, option type 등을 정상 결과처럼 허용하지 않는다.

## 7. 난수 책임과 재현성

난수 관련 책임은 두 수준으로 나뉜다.

- <code>sample_gbm_terminal()</code>은 이미 만들어진 Generator를 받는다. 이 함수는 전역 <code>np.random</code> 상태나 숨겨진 seed를 사용하지 않는다.
- <code>mc_price()</code>는 사용자가 지정한 seed로 새 Generator를 만든 뒤 sampler에 전달한다. 결과 객체에도 seed를 보존한다.

같은 모든 가격 입력과 같은 seed로 <code>mc_price()</code>를 다시 호출하면 같은 terminal samples와 같은 <code>MonteCarloResult</code>를 얻는다.

반면 $T>0$, $\sigma>0$인 확률적 분기에서 하나의 Generator 인스턴스를 sampler의 여러 호출에 계속 재사용하면 난수 stream 상태가 앞으로 진행하므로 각 호출의 표본은 달라진다. sampler 수준에서 동일 표본을 재현하려면 동일 seed로 Generator를 새로 생성해야 한다. $T=0$ 또는 $\sigma=0$인 결정론적 분기는 난수를 소비하지 않으므로 Generator 상태도 진행하지 않는다.

이 설계는 다음 두 목적을 동시에 만족한다.

1. 연구 노트북에서는 seed 하나로 실험을 간단하게 재현한다.
2. 더 낮은 수준의 simulation 코드는 호출자가 Generator stream을 직접 관리할 수 있다.

## 8. 입력 검증 구조

각 공개 함수는 자신의 계약을 독립적으로 검증한다. 특히 <code>mc_price()</code>는 $T=0$과 $\sigma=0$에서 sampler를 호출하지 않기 때문에 common GBM 입력도 직접 검증해야 한다.

| 검증 | 동작 |
|---|---|
| 수치 입력 | bool을 제외한 유한한 실수만 허용 |
| <code>spot</code>, <code>strike</code> | 0보다 커야 함 |
| <code>maturity</code>, <code>volatility</code> | 0 이상이어야 함 |
| <code>n_paths</code> | bool이 아닌 2 이상의 정수 |
| <code>option_type</code> | 문자열 <code>call</code> 또는 <code>put</code> |
| <code>seed</code> | bool이 아닌 0 이상의 정수 |
| <code>confidence_level</code> | 유한하고 열린 구간 $(0,1)$ 안에 있어야 함 |
| <code>rng</code> | <code>numpy.random.Generator</code> 인스턴스여야 함 |

무위험이자율과 배당수익률은 음수일 수 있으므로 유한성 외에 부호 제한을 두지 않는다. 잘못된 금융·수치 입력은 <code>ValueError</code>, 잘못된 Generator 타입은 <code>TypeError</code>를 발생시킨다. 입력값을 임의의 유효 범위로 clipping하거나 잘못된 값을 조용히 교체하지 않는다.

## 9. 테스트가 보호하는 계약

| 테스트 범주 | 확인하는 내용 |
|---|---|
| terminal sampler 재현성 | 같은 seed로 새 Generator를 만들면 배열 전체가 동일함 |
| terminal sampler 경계 | $T=0$ 배열과 $\sigma=0$ 결정론적 배열이 정확함 |
| pricer 재현성 | 같은 입력과 seed의 결과 객체가 동일함 |
| $T=0$ | call/put 내재가치, 표준오차 0, 퇴화 신뢰구간 |
| $\sigma=0$ | 할인된 결정론적 call/put payoff |
| 잘못된 입력 | <code>n_paths</code>, option type, finite 값, confidence level, seed 검증 |
| 신뢰구간 | 기본 95% 정규근사 임계값을 사용함 |
| <code>ddof=1</code> | 손으로 검산 가능한 두 payoff로 표본표준오차를 직접 확인 |
| BSM benchmark | 고정 seed MC 결과와 알려진 BSM 가격의 차이가 표준오차로 설명됨 |

<code>ddof=1</code> 테스트는 terminal prices를 $[80,120]$, strike를 100, rate를 0으로 고정한다. call의 할인 payoff는 $[0,20]$이고 평균은 10이다. 표본표준편차는

$$
s_Y=\sqrt{\frac{(0-10)^2+(20-10)^2}{2-1}}=\sqrt{200}
$$

이므로 표준오차는

$$
\frac{\sqrt{200}}{\sqrt2}=10
$$

이다. 이 테스트는 구현이 실수로 모집단 표준편차 <code>ddof=0</code>을 사용하면 실패한다.

BSM 검증은 패키지에 BSM 함수를 미리 구현하지 않고, 표준 입력의 고정 기준값을 테스트 상수로만 사용한다.

- call benchmark: 10.450583572185565
- put benchmark: 5.573526022256971
- $M=50{,}000$, seed 42
- 허용범위: $|\hat V_M-V_{\mathrm{BSM}}|\le4SE(\hat V_M)$

4-standard-error 허용범위는 정규근사에서 약 99.994% coverage에 해당하며, 고정 seed와 함께 통계적으로 설명 가능한 비-flaky 검증 기준을 만든다.

## 10. 연구 노트북과 tidy 결과 테이블

연구 노트북은 가격 공식을 다시 구현하지 않고 <code>mc_price()</code>만 import한다. CRR 노트북과 동일한 synthetic parameter를 사용한다.

$$
S=K=100,
\qquad T=1,
\qquad r=0.05,
\qquad q=0,
\qquad \sigma=0.2
$$

call과 put의 단일 예제 뒤에 여러 <code>n_paths</code> 값을 반복하여 다음 tidy-form DataFrame을 만든다.

| 열 | 의미 |
|---|---|
| <code>option_type</code> | call 또는 put |
| <code>n_paths</code> | 해당 추정의 경로 수 |
| <code>seed</code> | 재현 가능한 난수 seed |
| <code>mc_price</code> | <code>MonteCarloResult.price</code> |
| <code>standard_error</code> | 가격 추정의 표준오차 |
| <code>ci_low</code> | 95% 신뢰구간 하단 |
| <code>ci_high</code> | 95% 신뢰구간 상단 |

이 행 구조는 후속 BSM 구현 뒤 <code>bsm_price</code>, <code>absolute_error</code>, <code>relative_error</code> 열을 추가하고 Matplotlib에 바로 전달할 수 있다.

수렴 노트북은 각 <code>mc_price()</code> 호출에서 같은 seed로 Generator를 새로 만든다. 따라서 작은 <code>n_paths</code> 표본은 큰 <code>n_paths</code> 표본의 prefix이고, 같은 경로 수의 call과 put도 동일한 terminal paths를 사용하는 common-random-number 구조다. 결과 행들을 서로 독립적인 simulation으로 해석해서는 안 된다.

이론적으로 표준오차의 전형적 규모는 $1/\sqrt{M}$로 줄어들지만, nested sample에 새 경로가 추가될 때 표본평균은 위나 아래 어느 방향으로도 움직일 수 있다. 따라서 실제 추정가격과 BSM 대비 절대오차가 모든 <code>n_paths</code> 단계에서 단조롭게 감소하는 것은 아니다.

## 11. 현재 범위와 다음 단계

현재 구현에 포함된 범위는 다음과 같다.

- European call과 put
- 위험중립 GBM의 정확한 terminal distribution
- 벡터화된 terminal-price sampling
- 명시적 Generator 또는 seed 기반 재현성
- 할인 payoff 평균, <code>ddof=1</code> 표준오차, 정규근사 신뢰구간
- 수렴 실험용 tidy DataFrame

의도적으로 포함하지 않은 범위는 다음과 같다.

- 전체 GBM 시간경로와 Euler–Maruyama
- American, Asian, exotic option
- antithetic/control variates, quasi-Monte Carlo
- BSM 가격 함수와 최종 모델 비교 그래프
- Greeks, implied volatility, delta hedging

다음 단계에서는 별도의 BSM 구현을 추가한 뒤 동일한 입력에서 CRR과 GBM–Monte Carlo 결과를 BSM benchmark에 연결한다. 그때 CRR에는 discretization error, Monte Carlo에는 sampling error가 있다는 차이를 구분해 해석해야 한다.